In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from ocelescope import OCEL

In [2]:
ocel = OCEL.read_ocel(Path("./order-management.sqlite"))

In [13]:
def group_relation_entity(
    df: pd.DataFrame,
    entity_ids: list[str],
    id_column: str,
    type_column: str,
    target_id_column,
):
    return (
        df[df[id_column].isin(entity_ids)]
        .groupby([id_column, type_column])[target_id_column]
        .size()
        .reset_index(name="count")
        .rename(
            columns={
                id_column: "id",
                type_column: "type",
            }
        )
    )

In [45]:
from ocel_graph.resources.ocelGraph import Event, Object, OCELGraph, Relation

graph = OCELGraph()
depth = 3
max_neighbours = 10

root_id = "place_o-990001"
root = ocel.events[ocel.events[ocel.ocel.event_id_column] == root_id].iloc[0]
root_type = root[ocel.ocel.event_activity]

events_to_visit = [Event(id=root_id, activity_type=root_type)]
objects_to_visit = [Object(id="i-880001", object_type="items")]

event_ids_to_visit = [event.id for event in events_to_visit]
object_ids_to_visit = [object.id for object in objects_to_visit]

relations = ocel.relations[
    (
        (ocel.relations[ocel.ocel.event_id_column].isin(event_ids_to_visit))
        ^ (ocel.relations[ocel.ocel.object_id_column].isin(object_ids_to_visit))
    )
    & ~(ocel.relations[ocel.ocel.event_id_column].isin(graph.event_ids))
    & ~(ocel.relations[ocel.ocel.object_id_column].isin(graph.object_ids))
]

events = group_relation_entity(
    relations,
    event_ids_to_visit,
    ocel.ocel.event_id_column,
    ocel.ocel.event_activity,
    ocel.ocel.object_id_column,
)

e2o_objects = group_relation_entity(
    relations,
    object_ids_to_visit,
    ocel.ocel.object_id_column,
    ocel.ocel.object_type_column,
    ocel.ocel.event_id_column,
)

o2o = ocel.o2o[
    ((ocel.o2o["ocel:oid_1"].isin(object_ids_to_visit)) ^ (ocel.o2o["ocel:oid_2"].isin(object_ids_to_visit)))
    & ~(ocel.o2o["ocel:oid_1"].isin(graph.object_ids))
    & ~(ocel.o2o["ocel:oid_2"].isin(graph.object_ids))
]

o2o = o2o.rename(
    columns={"ocel:oid_1": "id", "ocel:type_1": "type", "ocel:oid_2": "target_id", "ocel:type_2": "target_type"}
)

mirrored = o2o.rename(columns={"target_id": "id", "target_type": "type", "id": "target_id", "type": "target_type"})

mirrored_o2o = pd.concat([o2o, mirrored], ignore_index=True).rename(columns={"ocel:qualifier": "qualifier"})


o2o_objects = group_relation_entity(
    mirrored_o2o,
    object_ids_to_visit,
    "id",
    "type",
    "target_id",
)

objects = (
    pd.concat([o2o_objects, e2o_objects], ignore_index=True).groupby(["id", "type"], as_index=False)["count"].sum()
)

In [46]:
graph.objects.update(objects_to_visit)
graph.events.update(events_to_visit)

objects_to_visit = []
events_to_visit = []

object_id_with_neighbours = [row["id"] for _, row in objects.iterrows() if row["count"] <= max_neighbours]
event_id_with_neighbours = [row["id"] for _, row in events.iterrows() if row["count"] <= max_neighbours]

for _, row in (
    mirrored_o2o[mirrored_o2o["id"].isin(object_id_with_neighbours)]
    .drop_duplicates(subset=["target_id"], keep="first")
    .iterrows()
):
    graph.o2oRelations.add(Relation(source=row["id"], target=row["target_id"], qualifier=row["qualifier"]))
    objects_to_visit.append(Object(id=row["target_id"], object_type=row["target_type"]))

for _, row in (
    relations[relations["ocel:oid"].isin(object_id_with_neighbours)]
    .drop_duplicates(subset=["ocel:eid"], keep="first")
    .iterrows()
):
    graph.e2oRelations.add(
        Relation(
            source=row["ocel:eid"],
            target=row["ocel:oid"],
            qualifier=row["ocel:qualifier"],
        )
    )
    events_to_visit.append(Event(id=row["ocel:eid"], activity_type=row["ocel:activity"]))

for _, row in (
    relations[
        relations["ocel:eid"].isin(event_id_with_neighbours)
        & ~relations["ocel:oid"].isin([object.id for object in objects_to_visit])
    ]
    .drop_duplicates(subset=["ocel:oid"], keep="first")
    .iterrows()
):
    graph.e2oRelations.add(
        Relation(
            source=row["ocel:eid"],
            target=row["ocel:oid"],
            qualifier=row["ocel:qualifier"],
        )
    )
    objects_to_visit.append(Object(id=row["ocel:oid"], object_type=row["ocel:type"]))

In [47]:
objects_to_visit

[Object(id='Echo', object_type='products'),
 Object(id='o-990001', object_type='orders'),
 Object(id='p-660001', object_type='packages'),
 Object(id='AlpenTech Innovations AG', object_type='customers'),
 Object(id='i-880002', object_type='items'),
 Object(id='i-880003', object_type='items'),
 Object(id='iPhone 11 Pro', object_type='products')]

In [48]:
events_to_visit

[Event(id='pick_i-880001', activity_type='pick item'),
 Event(id='confirm_o-990001', activity_type='confirm order'),
 Event(id='pay_o-990001', activity_type='pay order'),
 Event(id='create_p-660001', activity_type='create package'),
 Event(id='send_p-660001', activity_type='send package'),
 Event(id='deliver_p-660001', activity_type='package delivered')]